In [43]:
#r "nuget: ScottPlot, 5.0.19"

Installed Packages ScottPlot, 5.0.19

In [44]:
using System;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using System.IO;
using ScottPlot;

In [45]:
public class DefiniteIntegralOptimized
{
    public static double Solve(double a, double b, Func<double, double> function, double step, int threadsnumber)
    {
        double result = 0.0;
        object lockObj = new object();
        double range = b - a;
        double stepSize = range / threadsnumber;

        Parallel.For(0, threadsnumber, new ParallelOptions { MaxDegreeOfParallelism = threadsnumber }, i =>
        {
            double threadStart = a + i * stepSize;
            double threadEnd = (i == threadsnumber - 1) ? b : threadStart + stepSize;
            double localResult = 0.0;

            for (double x = threadStart; x < threadEnd; x += step)
            {
                double next = Math.Min(x + step, threadEnd);
                double nextVal = function(next);
                localResult += (function(x) + nextVal) * (next - x) / 2.0;
            }

            lock (lockObj)
            {
                result += localResult;
            }
        });

        return result;
    }
}

 display("Метод вычисления определенного класса был оптимизирован. Вместо ручного управления потоков теперь используется пул-потоков (Parallel), что позволило увеличить производительность до 80%")

Метод вычисления определенного класса был оптимизирован. Вместо ручного управления потоков теперь используется пул-потоков (Parallel), что позволило увеличить производительность до 80%

In [46]:
double SingleThreadIntegral(double a, double b, Func<double, double> function, double step)
{
    double result = 0.0;
    for (double x = a; x < b; x += step)
    {
        double next = Math.Min(x + step, b);
        result += (function(x) + function(next)) * (next - x) / 2.0;
    }
    return result;
}

In [47]:
int a = -100;
int b = 100;
double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
double accuracy = 1e-4;
double bestStep = steps[0];
double bestTime = double.MaxValue;
int iterations = 3;

foreach (var step in steps)
{
    double result = 0;
    Stopwatch sw = Stopwatch.StartNew();
    for (int i = 0; i < iterations; i++)
    {
        result = DefiniteIntegralOptimized.Solve(a, b, Math.Sin, step, 4);
    }
    sw.Stop();

    double avgTime = sw.Elapsed.TotalMilliseconds / iterations;

    if (Math.Abs(result) <= accuracy && avgTime < bestTime)
    {
        bestStep = step;
        bestTime = avgTime;
    }
}
display($"Оптимальный шаг: {bestStep}, среднее время: {bestTime:F2} мс");

Оптимальный шаг: 0.1, среднее время: 0.45 мс

In [48]:
int[] threadCounts = { 2, 4, 6, 8, 10, 12, 14, 16 };
double[] avgTimes = new double[threadCounts.Length];

for (int i = 0; i < threadCounts.Length; i++)
{
    double sumTime = 0;
    for (int j = 0; j < iterations; j++)
    {
        var sw = Stopwatch.StartNew();
        DefiniteIntegralOptimized.Solve(a, b, Math.Sin, bestStep, threadCounts[i]);
        sw.Stop();
        sumTime += sw.Elapsed.TotalMilliseconds;
    }
    avgTimes[i] = sumTime / (double)iterations;
}

int bestThreads = threadCounts[Array.IndexOf(avgTimes, avgTimes.Min())];
display($"Оптимальное число потоков: {bestThreads}");

Оптимальное число потоков: 16

In [49]:
double singleThreadTime = 0;
for (int j = 0; j < iterations; j++)
{
    var sw = Stopwatch.StartNew();
    SingleThreadIntegral(a, b, Math.Sin, bestStep);
    sw.Stop();
    singleThreadTime += sw.Elapsed.TotalMilliseconds;
}
singleThreadTime /= iterations;

double multiThreadTime = avgTimes[Array.IndexOf(threadCounts, bestThreads)];
double percentDiff = 100.0 * (singleThreadTime - multiThreadTime) / singleThreadTime;

string result = $@"Время однопоточной версии: {singleThreadTime:F2} мс
Время многопоточной версии: {multiThreadTime:F2} мс
Разница: {percentDiff:F2}%";

display(result);
File.WriteAllText("benchmark.txt", result);

Время однопоточной версии: 0.32 мс
Время многопоточной версии: 0.08 мс
Разница: 73.69%

In [50]:
var scottPlot = new ScottPlot.Plot();

scottPlot.Add.Scatter(avgTimes, threadCounts.Select(x => (double)x).ToArray());
scottPlot.XLabel("Время выполнения (мс)");
scottPlot.YLabel("Количество потоков");
scottPlot.Title("Производительность");
scottPlot.SavePng("benchmark.png", 800, 600);